In [1]:
import anndata, json
from pathlib import Path

# adjust to match your actual client folder naming/count
N_CLIENTS = 7  # from pyproject.toml: options.num-supernodes = 7
CLIENT_DIR_TEMPLATE = "../../../data_federated/data_client_{partition_id}"

rows = []
for pid in range(N_CLIENTS):
    client_folder = Path(CLIENT_DIR_TEMPLATE.format(partition_id=pid))

    # backed='r' avoids loading full expression matrices into memory,
    # just reads obs/shape metadata -- fast for a quick census
    train = anndata.read_h5ad(client_folder / "train.h5ad", backed="r")
    valid = anndata.read_h5ad(client_folder / "valid.h5ad", backed="r")

    techs_train = train.obs["tech"].value_counts().to_dict() if "tech" in train.obs else {}

    rows.append({
        "partition_id": pid,
        "n_train": train.n_obs,
        "n_valid": valid.n_obs,
        "n_total": train.n_obs + valid.n_obs,
        "tech_breakdown": techs_train,
    })

    train.file.close()
    valid.file.close()

import pandas as pd
df = pd.DataFrame(rows).sort_values("n_total")
pd.set_option("display.max_colwidth", None)
print(df.to_string(index=False))

 partition_id  n_train  n_valid  n_total      tech_breakdown
            1      510      128      638 {'fluidigmc1': 510}
            0      803      201     1004     {'celseq': 803}
            5     1042      261     1303   {'inDrop4': 1042}
            6     1194      298     1492   {'smarter': 1194}
            3     1379      345     1724   {'inDrop2': 1379}
            2     1550      387     1937   {'inDrop1': 1550}
            4     2884      721     3605   {'inDrop3': 2884}


In [2]:
# Which techs actually ended up as your 7 clients?
import anndata
from pathlib import Path

N_CLIENTS = 7
CLIENT_DIR_TEMPLATE = "../../../data_federated/data_client_{partition_id}"

for pid in range(N_CLIENTS):
    folder = Path(CLIENT_DIR_TEMPLATE.format(partition_id=pid))
    train = anndata.read_h5ad(folder / "train.h5ad", backed="r")
    print(pid, train.obs["tech"].unique().tolist(), train.n_obs)
    train.file.close()

0 ['celseq'] 803
1 ['fluidigmc1'] 510
2 ['inDrop1'] 1550
3 ['inDrop2'] 1379
4 ['inDrop3'] 2884
5 ['inDrop4'] 1042
6 ['smarter'] 1194


In [3]:
# Compare smarter vs. every other tech directly: library size + sparsity,
# using the pooled centralized data so it's apples-to-apples
import anndata, json
import numpy as np

pancreas_train = anndata.read_h5ad("../../../data_centralized/pancreas_train.h5ad")
hvg_list = json.load(open("../../../data_centralized/top2k_genes.json"))
pancreas_train_hvg = pancreas_train[:, hvg_list].copy()

for tech, sub in pancreas_train_hvg.obs.groupby("tech"):
    idx = (pancreas_train_hvg.obs["tech"] == tech).values
    X = pancreas_train_hvg[idx].layers["counts"] if "counts" in pancreas_train_hvg.layers else pancreas_train_hvg[idx].X
    total_counts = np.asarray(X.sum(axis=1)).ravel()
    pct_zero = 1.0 - (np.asarray((X > 0).sum(axis=1)).ravel() / X.shape[1])
    print(f"{tech:12s} n={idx.sum():5d}  "
          f"median_lib_size={np.median(total_counts):9.1f}  "
          f"median_pct_zero_genes={np.median(pct_zero):.3f}")

C:\Users\virgi\AppData\Local\Temp\ipykernel_19256\4033020226.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for tech, sub in pancreas_train_hvg.obs.groupby("tech"):


celseq       n=  803  median_lib_size=   1269.0  median_pct_zero_genes=0.869
fluidigmc1   n=  510  median_lib_size= 231108.6  median_pct_zero_genes=0.698
inDrop1      n= 1550  median_lib_size=    808.0  median_pct_zero_genes=0.935
inDrop2      n= 1379  median_lib_size=    878.0  median_pct_zero_genes=0.941
inDrop3      n= 2884  median_lib_size=    898.0  median_pct_zero_genes=0.937
inDrop4      n= 1042  median_lib_size=   1062.0  median_pct_zero_genes=0.927
smarter      n= 1194  median_lib_size= 167483.2  median_pct_zero_genes=0.808
